# Loss-function experiments - baseline U-Net on Task08 HepaticVessel

Same **baseline architecture** (networks/UNET.py, no dropout) - we change **only the loss** to improve raw segmentation (**Dice + ASSD**) on this highly imbalanced task (thin vessels + small tumours). Three runs share fold / seed / epochs for a fair comparison:

- **baseline** - `Dice + CE`
- **Combo 1** - `Focal-Tversky + CE + lambda*Boundary` (scheduled lambda) - recall of thin structures + surface/ASSD
- **Combo 2** - `clDice + Dice + CE` - topology / connectivity of tubular vessels

**Before running:** commit & push the new files (`train_losses.py`, `test_losses.py`, `loss_functions/{tversky,boundary,cldice}_loss.py`, `datasets/preprocessing_boundary.py`, `run_preprocessing_losses.py`) to the repo cloned below. Set the input paths in **init**. Run top to bottom.

In [ ]:
# ===== init =====
import os

# The (2+K)-channel preprocessed data (image, label, phi...) + splits.pkl live / are written here.
#   DO_PREPROCESS=True : point RAW_IN at the raw MSD task; we stage a writable dir and run the
#                        boundary preprocessing in this notebook.
#   DO_PREPROCESS=False: point PREP_IN at a dataset already preprocessed with run_preprocessing_losses.py.
DO_PREPROCESS = True
TASK          = 'Task08_HepaticVessel'

RAW_IN   = f'/kaggle/input/task08-hepaticvessel/{TASK}'         # raw: imagesTr/ labelsTr/ dataset.json
PREP_IN  = f'/kaggle/input/task08-hepaticvessel-losses/{TASK}'  # already-preprocessed (DO_PREPROCESS=False)
WORK_DIR = f'/kaggle/working/data/{TASK}'                       # writable target for preprocessing

DATA_DIR = WORK_DIR if DO_PREPROCESS else PREP_IN
os.environ['DATA_DIR'] = DATA_DIR
os.environ['TASK']     = TASK

# experiment config
FOLDS    = [0]           # e.g. [0, 1, 2, 3, 4] for full 5-fold CV
EPOCHS   = 150
PATIENCE = 12
OUT_DIR  = 'results'     # inside the repo -> persists as Kaggle output

# loss hyper-params
TVERSKY_ALPHA, TVERSKY_BETA, FOCAL_GAMMA = 0.3, 0.7, 1.3333333
BOUNDARY_MAX, BOUNDARY_WARMUP            = 0.5, 40
CLDICE_WEIGHT, CLDICE_ITERS              = 0.5, 10

In [ ]:
# ===== repo + requirements =====
!rm -rf repo
!git clone 'https://github.com/ThanoSnake/my_unet.git' repo
%cd repo
!pip install -q medpy batchgenerators==0.21

In [ ]:
# ===== setup =====
import os, shutil

# (a) Kaggle ships HuggingFace 'datasets'; our local datasets/ needs __init__.py to win on sys.path.
for d in ['datasets','datasets/two_dim','datasets/three_dim','networks','loss_functions','evaluation','utilities']:
    os.makedirs(d, exist_ok=True)
    open(os.path.join(d, '__init__.py'), 'a').close()

# (b) stage a WRITABLE task dir BEFORE importing config (config reads dataset.json for modality/labels).
#     imagesTr/labelsTr are symlinked (read-only, no big copy); only dataset.json is copied.
if DO_PREPROCESS:
    os.makedirs(WORK_DIR, exist_ok=True)
    for sub in ['imagesTr','labelsTr']:
        dst = os.path.join(WORK_DIR, sub)
        if not os.path.exists(dst):
            os.symlink(os.path.join(RAW_IN, sub), dst)
    shutil.copy(os.path.join(RAW_IN, 'dataset.json'), os.path.join(WORK_DIR, 'dataset.json'))

import config
print('TASK:', config.TASK, '| classes:', config.NUM_CLASSES, '| labels:', config.LABELS)
print('modality:', config.MODALITY, '| DATA_DIR:', config.DATA_DIR)

In [ ]:
# ===== preprocessing -> (2+K)-channel npy [image, label, phi_1..phi_K] + splits.pkl =====
# clDice needs no precompute; the phi maps are only for the boundary term. Runs once.
if DO_PREPROCESS:
    !python run_preprocessing_losses.py
else:
    print('Skipping preprocessing; using', config.DATA_DIR)

In [ ]:
# ===== verify data (image + label + K distance maps) =====
import glob, numpy as np
assert config.PREPROCESSED_DIR.exists(), config.PREPROCESSED_DIR
assert config.SPLITS_FILE.exists(), config.SPLITS_FILE
one = sorted(glob.glob(str(config.PREPROCESSED_DIR / '*.npy')))[0]
ch = np.load(one, mmap_mode='r').shape[0]
need = 2 + (config.NUM_CLASSES - 1)      # image + label + K signed-distance maps
print(f'{os.path.basename(one)}: {ch} channels (need >= {need}: image, label, {config.NUM_CLASSES-1} SDF)')
assert ch >= need, 'distance-map channels missing -> re-run run_preprocessing_losses.py'

## Train + test the three models
Each `train_losses.py` saves `<tag>_f<fold>_best.pth`; `test_losses.py` writes `<tag>_f<fold>_scores.json` (Dice, ASSD, ... per class) using the untouched `evaluate_test`. Same `--epochs` / `--patience` / seed for all three -> fair comparison.

In [ ]:
# ===== baseline: Dice + CE =====
for f in FOLDS:
    !python train_losses.py --loss dice_ce --fold {f} --epochs {EPOCHS} --patience {PATIENCE} --out-dir {OUT_DIR} --num-workers 0
    !python test_losses.py  --tag  dice_ce --fold {f} --out-dir {OUT_DIR} --num-workers 0

In [ ]:
# ===== Combo 1: Focal-Tversky + CE + lambda*Boundary (scheduled) =====
for f in FOLDS:
    !python train_losses.py --loss ftversky_ce_boundary --fold {f} --epochs {EPOCHS} --patience {PATIENCE} --tversky-alpha {TVERSKY_ALPHA} --tversky-beta {TVERSKY_BETA} --focal-gamma {FOCAL_GAMMA} --boundary-max {BOUNDARY_MAX} --boundary-warmup {BOUNDARY_WARMUP} --out-dir {OUT_DIR} --num-workers 0
    !python test_losses.py  --tag  ftversky_ce_boundary --fold {f} --out-dir {OUT_DIR} --num-workers 0

In [ ]:
# ===== Combo 2: clDice + Dice + CE =====
for f in FOLDS:
    !python train_losses.py --loss cldice_dice_ce --fold {f} --epochs {EPOCHS} --patience {PATIENCE} --cldice-weight {CLDICE_WEIGHT} --cldice-iters {CLDICE_ITERS} --out-dir {OUT_DIR} --num-workers 0
    !python test_losses.py  --tag  cldice_dice_ce --fold {f} --out-dir {OUT_DIR} --num-workers 0

## Compare - Dice (higher=better) and ASSD (lower=better) per class
Mean over folds, with the delta vs the `dice_ce` baseline (for ASSD a negative delta = better).

In [ ]:
# ===== compare Dice / ASSD per class across the three models =====
import os, json
import numpy as np

MODELS  = ['dice_ce', 'ftversky_ce_boundary', 'cldice_dice_ce']
METRICS = ['Dice', 'Avg. Symmetric Surface Distance']

def collect(tag):
    per_label = {}
    for f in FOLDS:
        p = os.path.join(OUT_DIR, f'{tag}_f{f}_scores.json')
        if not os.path.exists(p):
            continue
        mean = json.load(open(p))['results']['mean']
        for label, md in mean.items():
            if label in ('test', 'reference'):
                continue
            per_label.setdefault(label, {m: [] for m in METRICS})
            for m in METRICS:
                v = md.get(m)
                if v is not None:
                    per_label[label][m].append(v)
    return {lab: {m: (float(np.nanmean(vs)) if vs else None) for m, vs in d.items()}
            for lab, d in per_label.items()}

results = {t: collect(t) for t in MODELS}
labels  = sorted({lab for r in results.values() for lab in r})

for m in METRICS:
    arrow = 'lower=better' if 'Distance' in m else 'higher=better'
    print()
    print(f'=== {m}  ({arrow}) ===')
    print('label'.ljust(12) + ''.join(t.ljust(26) for t in MODELS))
    for lab in labels:
        row  = str(lab).ljust(12)
        base = results[MODELS[0]].get(lab, {}).get(m)
        for t in MODELS:
            v = results[t].get(lab, {}).get(m)
            if v is None:
                cell = '-'
            elif t == MODELS[0] or base is None:
                cell = f'{v:.4f}'
            else:
                cell = f'{v:.4f} ({v-base:+.4f})'
            row += cell.ljust(26)
        print(row)

## Bar charts - Dice and ASSD per class, three models

In [ ]:
# ===== grouped bar charts: Dice and ASSD per class, three models =====
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, len(METRICS), figsize=(6.5 * len(METRICS), 4.3))
axes = np.atleast_1d(axes)
x = np.arange(len(labels))
w = 0.8 / len(MODELS)
for ax, m in zip(axes, METRICS):
    for i, t in enumerate(MODELS):
        vals = [results[t].get(lab, {}).get(m) or np.nan for lab in labels]
        ax.bar(x + i * w, vals, width=w, label=t)
    ax.set_xticks(x + w * (len(MODELS) - 1) / 2)
    ax.set_xticklabels(labels)
    ax.set_title(m)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()